 Gini impurity and entropy


In [3]:
import math 

def gini_impurity(labels):
    n=len(labels)
    if n==0:
        return 0.0
    counts={}
    for label in labels:
        counts[label]=counts.get(label,0)+1
    return 1.0 - sum((c/n)**2 for c in counts.values())
        

def entropy(labels):
    n = len(labels)
    if n == 0:
        return 0.0
    counts = {}
    for label in labels:
        counts[label] = counts.get(label, 0) + 1
    return -sum((c/n) * math.log2(c/n) for c in counts.values() if c > 0)

In [4]:
# ---- Test 1: pure (only one class) ----
labels1 = ['cat', 'cat', 'cat', 'cat']
print("Test 1:", labels1)
print("  Gini   =", gini_impurity(labels1))   # 0.0
print("  Entropy=", entropy(labels1))         # 0.0

# ---- Test 2: 50/50 split ----
labels2 = ['cat', 'dog', 'cat', 'dog']
print("\nTest 2:", labels2)
print("  Gini   =", gini_impurity(labels2))   # 0.5
print("  Entropy=", entropy(labels2))         # 1.0

# ---- Test 3: 3 cats, 2 dogs ----
labels3 = ['cat', 'dog', 'cat', 'cat', 'dog']
print("\nTest 3:", labels3)
print("  Gini   =", round(gini_impurity(labels3), 4))  # 0.48
print("  Entropy=", round(entropy(labels3), 4))        # 0.971

# ---- Test 4: empty list ----
print("\nTest 4: []")
print("  Gini   =", gini_impurity([]))         # 0.0
print("  Entropy=", entropy([]))               # 0.0

# ---- Test 5: counting trace ----
print("\n--- counting trace ---")
labels = ['a', 'b', 'a', 'c', 'a', 'b']
counts = {}
for label in labels:
    counts[label] = counts.get(label, 0) + 1
    print(f"  after seeing {label!r}: {counts}")

Test 1: ['cat', 'cat', 'cat', 'cat']
  Gini   = 0.0
  Entropy= -0.0

Test 2: ['cat', 'dog', 'cat', 'dog']
  Gini   = 0.5
  Entropy= 1.0

Test 3: ['cat', 'dog', 'cat', 'cat', 'dog']
  Gini   = 0.48
  Entropy= 0.971

Test 4: []
  Gini   = 0.0
  Entropy= 0.0

--- counting trace ---
  after seeing 'a': {'a': 1}
  after seeing 'b': {'a': 1, 'b': 1}
  after seeing 'a': {'a': 2, 'b': 1}
  after seeing 'c': {'a': 2, 'b': 1, 'c': 1}
  after seeing 'a': {'a': 3, 'b': 1, 'c': 1}
  after seeing 'b': {'a': 3, 'b': 2, 'c': 1}


Find the best split

In [6]:
def information_gain(parent_labels,left_labels,right_labels,criterion="gini"):
    measure=gini_impurity if criterion=="gini" else entropy
    n=len(parent_labels)
    n_left=len(left_labels)
    n_right=len(right_labels)
    if n_left==0 or n_right==0:
        return 0.0
    parent_impurity=measure(parent_labels)
    child_impurity = (
        (n_left / n) * measure(left_labels) +
        (n_right / n) * measure(right_labels)
    )
    return parent_impurity-child_impurity


In [7]:
# ---- Test 1: PERFECT split (Gini) ----
# Parent: 2 cats + 2 dogs (mixed). Split puts all cats left, all dogs right.
parent = ['cat', 'cat', 'dog', 'dog']
left   = ['cat', 'cat']
right  = ['dog', 'dog']
print("Test 1 (perfect split, gini):", information_gain(parent, left, right, "gini"))
# parent Gini = 0.5, both children pure (0). IG = 0.5 - 0 = 0.5


# ---- Test 1b: PERFECT split (entropy) ----
print("Test 1b (perfect split, entropy):", information_gain(parent, left, right, "entropy"))
# parent H = 1.0, both children 0. IG = 1.0 - 0 = 1.0


# ---- Test 2: NO-OP split (all children still mixed) ----
# Parent same, but split puts a cat+dog on each side → no reduction.
left2  = ['cat', 'dog']
right2 = ['cat', 'dog']
print("Test 2 (useless split, gini):", information_gain(parent, left2, right2, "gini"))
# parent Gini = 0.5, children Gini = 0.5 each, weighted = 0.5. IG = 0.0


# ---- Test 3: DEGENERATE split (one side empty) ----
left3  = ['cat', 'cat', 'dog', 'dog']
right3 = []
print("Test 3 (empty child):", information_gain(parent, left3, right3, "gini"))
# returns 0.0 by the guard clause


# ---- Test 4: UNBALANCED but still useful split ----
# Parent: 3 cats + 1 dog. Left gets 2 cats + 1 dog, right gets 1 cat.
parent4 = ['cat', 'cat', 'cat', 'dog']
left4   = ['cat', 'cat', 'dog']
right4  = ['cat']
print("Test 4 (unbalanced, gini):", round(information_gain(parent4, left4, right4, "gini"), 4))
# parent Gini   = 1 - (3/4)^2 - (1/4)^2 = 0.375
# left Gini     = 1 - (2/3)^2 - (1/3)^2 = 0.4444
# right Gini    = 0
# weighted child = (3/4)*0.4444 + (1/4)*0 = 0.3333
# IG = 0.375 - 0.3333 = 0.0417


# ---- Test 5: UNBALANCED split (entropy) ----
print("Test 5 (unbalanced, entropy):", round(information_gain(parent4, left4, right4, "entropy"), 4))
# parent H       = -(0.75*log2 0.75 + 0.25*log2 0.25) = 0.8113
# left H         = -(2/3*log2 2/3 + 1/3*log2 1/3)     = 0.9183
# right H        = 0
# weighted child = (3/4)*0.9183 = 0.6887
# IG = 0.8113 - 0.6887 = 0.1226


# ---- Test 6: NO GAIN at all (split a pure parent) ----
parent6 = ['cat', 'cat', 'cat']
left6   = ['cat', 'cat']
right6  = ['cat']
print("Test 6 (pure parent, gini):", information_gain(parent6, left6, right6, "gini"))
# parent Gini = 0, both children 0, IG = 0.0


# ---- Test 7: NO GAIN at all (pure parent, entropy) ----
print("Test 7 (pure parent, entropy):", information_gain(parent6, left6, right6, "entropy"))
# parent H = 0, IG = 0.0

Test 1 (perfect split, gini): 0.5
Test 1b (perfect split, entropy): 1.0
Test 2 (useless split, gini): 0.0
Test 3 (empty child): 0.0
Test 4 (unbalanced, gini): 0.0417
Test 5 (unbalanced, entropy): 0.1226
Test 6 (pure parent, gini): 0.0
Test 7 (pure parent, entropy): 0.0


Build the DecisionTree class

In [8]:
import random

class DecisionTree:
    def __init__(self, max_depth=None, min_samples_split=2,
                 min_samples_leaf=1, criterion="gini",
                 max_features=None):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.criterion = criterion
        self.max_features = max_features
        self.tree = None
        self.feature_importances_ = None
        
    def fit(self,X,y):
        self.n_features=len(X[0])
        self.feature_importances_=[0.0]*self.n_features
        self.n_samples=len(X)
        self.tree=self._build(X,y,depth=0)
        total = sum(self.feature_importances_)
        if total > 0:
            self.feature_importances_ = [
                fi / total for fi in self.feature_importances_
            ]
    
    
    def predict(self, X):
        return [self._predict_one(x, self.tree) for x in X]      
    
    def _build(self, X, y, depth):
        if len(set(y)) == 1:
            return {"leaf": True, "value": y[0]}

        if self.max_depth is not None and depth >= self.max_depth:
            return self._make_leaf(y)

        if len(y) < self.min_samples_split:
            return self._make_leaf(y)

        best_feature, best_threshold, best_gain = self._best_split(X, y)

        if best_feature is None or best_gain <= 0:
            return self._make_leaf(y)

        left_X, left_y, right_X, right_y = self._split_data(
            X, y, best_feature, best_threshold
        )

        if len(left_y) < self.min_samples_leaf or len(right_y) < self.min_samples_leaf:
            return self._make_leaf(y)

        weight = len(y) / self.n_samples
        self.feature_importances_[best_feature] += weight * best_gain

        return {
            "leaf": False,
            "feature": best_feature,
            "threshold": best_threshold,
            "left": self._build(left_X, left_y, depth + 1),
            "right": self._build(right_X, right_y, depth + 1),
        }

    def _make_leaf(self, y):
        counts = {}
        for label in y:
            counts[label] = counts.get(label, 0) + 1
        return {"leaf": True, "value": max(counts, key=counts.get)}

    def _best_split(self, X, y):
        best_feature = None
        best_threshold = None
        best_gain = -1.0

        if self.max_features == "sqrt":
            k = max(1, int(math.sqrt(self.n_features)))
            feature_indices = random.sample(range(self.n_features), k)
        elif isinstance(self.max_features, int):
            if self.max_features < 1:
                raise ValueError("max_features must be at least 1 when given as an integer")
            k = min(self.max_features, self.n_features)
            feature_indices = random.sample(range(self.n_features), k)
        else:
            feature_indices = list(range(self.n_features))

        for feature_idx in feature_indices:
            values = sorted(set(X[i][feature_idx] for i in range(len(X))))
            if len(values) <= 1:
                continue

            for i in range(len(values) - 1):
                threshold = (values[i] + values[i + 1]) / 2.0
                left_y = [y[j] for j in range(len(X)) if X[j][feature_idx] <= threshold]
                right_y = [y[j] for j in range(len(X)) if X[j][feature_idx] > threshold]

                if len(left_y) < self.min_samples_leaf or len(right_y) < self.min_samples_leaf:
                    continue

                gain = information_gain(y, left_y, right_y, self.criterion)
                if gain > best_gain:
                    best_gain = gain
                    best_feature = feature_idx
                    best_threshold = threshold

        return best_feature, best_threshold, best_gain

    def _split_data(self, X, y, feature, threshold):
        left_X, left_y, right_X, right_y = [], [], [], []
        for i in range(len(X)):
            if X[i][feature] <= threshold:
                left_X.append(X[i])
                left_y.append(y[i])
            else:
                right_X.append(X[i])
                right_y.append(y[i])
        return left_X, left_y, right_X, right_y

    def _predict_one(self, x, node):
        if node["leaf"]:
            return node["value"]
        if x[node["feature"]] <= node["threshold"]:
            return self._predict_one(x, node["left"])
        return self._predict_one(x, node["right"])
    

In [9]:
X = [
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1],
]
y = [0, 1, 1, 0]

print("=== Training XOR ===")
tree = DecisionTree(max_depth=3, criterion="gini")
tree.fit(X, y)

print("Predictions on training X:", tree.predict(X))
print("Expected:               ", y)
print("Feature importances:    ", [round(fi, 3) for fi in tree.feature_importances_])
print()


# ---- Manual verification of the first split ----
# The tree has 4 points, labels [0,1,1,0]. Both features are symmetric.
# Gini(parent) = 1 - (2/4)^2 - (2/4)^2 = 0.5
print("Gini(parent) =", gini_impurity(y))     # 0.5

# Consider splitting feature 0 at threshold 0.5:
#   left  (x1<=0.5): points (0,0),(0,1) -> labels [0,1]
#   right (x1>0.5) : points (1,0),(1,1) -> labels [1,0]
left_y  = [y[i] for i in range(4) if X[i][0] <= 0.5]
right_y = [y[i] for i in range(4) if X[i][0] >  0.5]
print("Split feat0 @ 0.5 -> left labels:", left_y, " right:", right_y)
print("  IG =", information_gain(y, left_y, right_y, "gini"))   # 0.5 - 0.5 = 0.0

# Consider feature 1 at 0.5 — by symmetry, identical (IG=0).
# So the FIRST split has IG=0. But the tree still splits (gain > 0 required?
# let's recheck) — IG here = 0.5 - (0.5*1.0 + 0.5*1.0) = 0.0
# So best_gain = 0.0 at the root. This is XOR — no single split helps!


# =======================
# Test 2: Linearly separable — should hit 100%
# =======================
# Label = 1 if x1 > 0.5 else 0. One split solves it.
X2 = [[1],[2],[3],[4],[5],[6]]
y2 = [0, 0, 0, 1, 1, 1]

print("\n=== Linearly separable ===")
tree2 = DecisionTree(max_depth=3, criterion="gini")
tree2.fit(X2, y2)
print("Predictions:", tree2.predict(X2))
print("Expected:   ", y2)


# =======================
# Test 3: Entropy criterion should give SAME predictions
# =======================
print("\n=== Same data, entropy criterion ===")
tree3 = DecisionTree(max_depth=3, criterion="entropy")
tree3.fit(X2, y2)
print("Predictions:", tree3.predict(X2))


# =======================
# Test 4: max_depth=1 vs max_depth=5 on noisy data
# =======================
X4 = [[i] for i in range(10)]
y4 = [0,0,1,0,1,1,0,1,1,0]     # noisy alternation

print("\n=== max_depth=1 (underfit) ===")
t = DecisionTree(max_depth=1)
t.fit(X4, y4)
print("Preds:", t.predict(X4))

print("=== max_depth=5 (overfit — memorizes training set) ===")
t = DecisionTree(max_depth=5)
t.fit(X4, y4)
print("Preds:", t.predict(X4))
print("True :", y4)


# =======================
# Test 5: What happens with max_features="sqrt"
# =======================
print("\n=== max_features='sqrt' on XOR (random feature subset) ===")
tree5 = DecisionTree(max_depth=3, max_features="sqrt")
tree5.fit(X, y)
print("Predictions:", tree5.predict(X))
print("Importances:", [round(fi, 3) for fi in tree5.feature_importances_])

=== Training XOR ===
Predictions on training X: [0, 0, 0, 0]
Expected:                [0, 1, 1, 0]
Feature importances:     [0.0, 0.0]

Gini(parent) = 0.5
Split feat0 @ 0.5 -> left labels: [0, 1]  right: [1, 0]
  IG = 0.0

=== Linearly separable ===
Predictions: [0, 0, 0, 1, 1, 1]
Expected:    [0, 0, 0, 1, 1, 1]

=== Same data, entropy criterion ===
Predictions: [0, 0, 0, 1, 1, 1]

=== max_depth=1 (underfit) ===
Preds: [0, 0, 1, 1, 1, 1, 1, 1, 1, 1]
=== max_depth=5 (overfit — memorizes training set) ===
Preds: [0, 0, 1, 1, 1, 1, 0, 1, 1, 0]
True : [0, 0, 1, 0, 1, 1, 0, 1, 1, 0]

=== max_features='sqrt' on XOR (random feature subset) ===
Predictions: [0, 0, 0, 0]
Importances: [0.0, 0.0]


Build the RandomForest class

In [10]:
class RandomForest:
    def __init__(self, n_trees=100, max_depth=None,
                 min_samples_split=2, max_features="sqrt",
                 criterion="gini"):
        self.n_trees = n_trees
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_features = max_features
        self.criterion = criterion
        self.trees = []

    def fit(self, X, y):
        n = len(X)
        for _ in range(self.n_trees):
            indices = [random.randint(0, n - 1) for _ in range(n)]
            X_boot = [X[i] for i in indices]
            y_boot = [y[i] for i in indices]
            tree = DecisionTree(
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split,
                max_features=self.max_features,
                criterion=self.criterion,
            )
            tree.fit(X_boot, y_boot)
            self.trees.append(tree)

    def predict(self, X):
        all_preds = [tree.predict(X) for tree in self.trees]
        predictions = []
        for i in range(len(X)):
            votes = {}
            for preds in all_preds:
                v = preds[i]
                votes[v] = votes.get(v, 0) + 1
            predictions.append(max(votes, key=votes.get))
        return predictions

In [11]:
X = [[1],[2],[3],[4],[5],[6],[7],[8]]
y = [0, 0, 0, 0, 1, 1, 1, 1]     # clean threshold at 4.5

print("=== Test 1: cleanly separable data ===")
rf = RandomForest(n_trees=5, max_depth=3, max_features=1)
rf.fit(X, y)
print("Predictions:", rf.predict(X))
print("Expected   :", y)


# ---- Noisy dataset — forest should still do well ----
X2 = [[i] for i in range(10)]
y2 = [0, 0, 1, 0, 1, 1, 0, 1, 1, 0]   # noisy alternation

print("\n=== Test 2: noisy data ===")
rf2 = RandomForest(n_trees=10, max_depth=3, max_features=1)
rf2.fit(X2, y2)
preds = rf2.predict(X2)
print("Predictions:", preds)
print("True       :", y2)
print("Accuracy   :", sum(p == t for p, t in zip(preds, y2)) / len(y2))


# ---- Show that trees are actually different (bootstrap randomness) ----
print("\n=== Test 3: trees make different predictions (that's the point) ===")
for i, tree in enumerate(rf2.trees):
    print(f"  tree {i}: {tree.predict(X2)}")

=== Test 1: cleanly separable data ===
Predictions: [0, 0, 0, 0, 1, 1, 1, 1]
Expected   : [0, 0, 0, 0, 1, 1, 1, 1]

=== Test 2: noisy data ===
Predictions: [0, 0, 1, 0, 1, 1, 0, 0, 1, 0]
True       : [0, 0, 1, 0, 1, 1, 0, 1, 1, 0]
Accuracy   : 0.9

=== Test 3: trees make different predictions (that's the point) ===
  tree 0: [0, 0, 0, 0, 1, 1, 0, 0, 0, 0]
  tree 1: [0, 0, 1, 0, 0, 0, 0, 0, 1, 0]
  tree 2: [0, 0, 1, 1, 1, 1, 1, 1, 1, 0]
  tree 3: [0, 0, 0, 0, 1, 1, 1, 1, 1, 0]
  tree 4: [0, 0, 0, 0, 1, 1, 0, 0, 0, 0]
  tree 5: [0, 0, 1, 0, 0, 0, 0, 0, 0, 0]
  tree 6: [0, 0, 1, 1, 1, 1, 0, 1, 1, 1]
  tree 7: [0, 0, 1, 1, 1, 1, 0, 0, 0, 0]
  tree 8: [0, 0, 1, 0, 1, 1, 1, 1, 1, 1]
  tree 9: [1, 1, 1, 0, 0, 0, 0, 1, 1, 0]
